# Rhode Island — Title 27 (Insurance) → `data/rhode_island/ins_codes/*.md`

Rhode Island’s **insurance** statutes are **Title 27 — Insurance** of the **Rhode Island General Laws (R.I. Gen. Laws)**. On **Justia**, the crawl root is **[`/codes/rhode-island/title-27/`](https://law.justia.com/codes/rhode-island/title-27/)**; sections live under nested **`chapter-27-…`** paths, e.g. **`…/chapter-27-1-1/section-27-1-1-1/`** (BFS over chapter index pages, same idea as **`ohio.ipynb`** / **`pennsylvania.ipynb`**).

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** BFS from the Title 27 index, following only paths under **`/codes/rhode-island/title-27/`** that are **not** section pages, skipping common non-statute paths (**`appendix`**, **`chronological-history`**, **`title-notes`**, **`historical`**); collect every section link (~**2,219** sections across ~**167** chapter pages).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`RI_sec_<slug>.md`** where **`<slug>`** normalizes the label after **`section-`** (e.g. `27-1-1-1` → `RI_sec_27_1_1_1.md`). Display cites **`R.I. Gen. Laws § …`** using that same label.

Config: **MAX_SECTIONS** (**0** = all), **MAX_DISCOVERY_PAGES** (**0** = no cap). **REUSE_DISCOVERED_URLS** skips discovery when **`_ri_title27_section_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/rhode-island/title-27"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "rhode_island" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_ri_title27_section_urls.txt"
REUSE_DISCOVERED_URLS = True

SKIP_PATH_SUBSTR = ("appendix", "chronological-history", "title-notes", "historical")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def skip_path(p: str) -> bool:
    low = p.lower()
    return any(s in low for s in SKIP_PATH_SUBSTR)


def discover_section_urls() -> list[str]:
    """BFS title-27 index + nested chapter pages; collect section URLs."""
    from collections import deque

    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start).lower()}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url).lower()
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-" in pk.lower():
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        if fetches % 25 == 0:
            print(f"… discovery fetch {fetches}, queue={len(q)}, sections={len(sections)}")
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu).lower()
            if not p.startswith(PATH_PREFIX):
                continue
            if skip_path(p):
                continue
            if "/section-" in p.lower():
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")
    return sorted(sections, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    return path.rsplit("/section-", 1)[1]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    return f"R.I. Gen. Laws § {label}"


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"RI_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("R.I. Gen" in s or "Rhode Island General" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("R.I. Gen. Laws"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title_27() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 27")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Rhode Island General Laws {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Rhode Island General Laws — Title 27 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [Rhode Island General Laws (RILIN)](http://webserver.rilin.state.ri.us/Statutes/)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title_27()


… discovery fetch 25, queue=142, sections=320
… discovery fetch 50, queue=117, sections=687
… discovery fetch 75, queue=92, sections=1069
… discovery fetch 100, queue=67, sections=1474
… discovery fetch 125, queue=42, sections=1872
… discovery fetch 150, queue=17, sections=2072
Discovered 2219 section URLs under Title 27
… 200/2219 (wrote=196 skipped=4 failed=0)
… 400/2219 (wrote=393 skipped=7 failed=0)
… 600/2219 (wrote=591 skipped=9 failed=0)
… 800/2219 (wrote=789 skipped=11 failed=0)
… 1000/2219 (wrote=982 skipped=18 failed=0)
… 1200/2219 (wrote=1181 skipped=19 failed=0)
… 1400/2219 (wrote=1377 skipped=23 failed=0)
… 1600/2219 (wrote=1577 skipped=23 failed=0)
… 1800/2219 (wrote=1777 skipped=23 failed=0)
… 2000/2219 (wrote=1977 skipped=23 failed=0)
… 2200/2219 (wrote=2177 skipped=23 failed=0)
Done. wrote=2196 skipped=23 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/rhode_island/ins_codes


{'wrote': 2196, 'skipped': 23, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
